In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "huggingface_hub<1.0", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import numpy as np
import torch

from data.loaders import get_data_loaders
from evaluation.metrics import evaluate_probe
from evaluation.palm import format_palm_report, palm_evaluate
from features.visual import DINOv2Extractor, extract_image_features
from features.vlm import vlm_feature_cache_paths
from training.checkpoint import load_probe
from utils import set_seed
from utils.kaggle import find_data_root, find_vlm_cache

In [ ]:
DATA_ROOT = find_data_root([
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
DATASET = "pathmnist"  # pathmnist | histoset | skintissue
SEED = 42  # any int; one seed per run

RUN_NAMES = [
    "scalpel_disagreement",
    "scalpel_visual_margin",
    "uncertainty_herding",
]

CHECKPOINT_ROOT = "/kaggle/input/EDIT_RUN_OUTPUT_SLUG/checkpoints"

VLM_FEATURE_DIR = "/kaggle/input/EDIT_VLM_CACHE_SLUG/vlm_features"

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

device = torch.device(config["device"])
data_path = Path(DATA_PATHS[DATASET])
checkpoint_dir = Path(CHECKPOINT_ROOT) / DATASET
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert checkpoint_dir.is_dir(), f"Missing checkpoint directory: {checkpoint_dir}"

set_seed(SEED)
_, test_loader, _ = get_data_loaders(str(data_path), SEED, verbose=True)
test_dataset = test_loader.dataset
test_labels = (
    test_dataset.lbl
    if hasattr(test_dataset, "lbl")
    else np.array(test_dataset.dataset.targets)[test_dataset.indices]
)

In [ ]:
budgets = config["cumulative_budget"]
run_encoder = {}
run_encoder_kind = {}

for run_name in RUN_NAMES:
    probe_paths = [
        checkpoint_dir / f"{run_name}_probe_budget_{budget}.pt" for budget in budgets
    ]
    missing = [p for p in probe_paths if not p.is_file()]
    assert not missing, f"{run_name}: missing checkpoints {missing}"

    seen = set()
    for path in probe_paths:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        metadata = checkpoint.get("metadata", {})
        seen.add((
            metadata.get("encoder", "facebook/dinov2-base"),
            metadata.get("encoder_kind", "dinov2"),
        ))
    assert len(seen) == 1, (
        f"{run_name}: budgets disagree on which encoder produced them "
        f"({seen}) -- this run's checkpoints are not a consistent sweep"
    )
    run_encoder[run_name], run_encoder_kind[run_name] = seen.pop()

encoders_used = sorted(set(run_encoder.values()))
print("encoder per run:")
for run_name in RUN_NAMES:
    print(f"  {run_name}: {run_encoder[run_name]} ({run_encoder_kind[run_name]})")
print(f"\ndistinct encoders in this comparison: {encoders_used}")

In [ ]:
encoder_kind_by_name = {}
for run_name in RUN_NAMES:
    encoder_kind_by_name[run_encoder[run_name]] = run_encoder_kind[run_name]

test_features_by_encoder = {}

for encoder in encoders_used:
    kind = encoder_kind_by_name[encoder]
    if kind == "dinov2":
        extractor = DINOv2Extractor(model_name=encoder).to(device)
        features = extract_image_features(test_loader, extractor, device)
        del extractor
        test_features_by_encoder[encoder] = features
        print(f"[dinov2] {encoder}: test features {features.shape}")
    elif kind == "vlm":
        vlm_dir = find_vlm_cache(DATASET, SEED, encoder, hint=VLM_FEATURE_DIR)
        assert vlm_dir is not None, (
            f"No VLM feature cache found for encoder={encoder!r} -- attach the "
            "Kaggle Dataset extract_vlm_features.ipynb published and set "
            "VLM_FEATURE_DIR to it (or a directory find_vlm_cache can search under)."
        )
        paths = vlm_feature_cache_paths(str(vlm_dir), DATASET, SEED, encoder)
        features = np.load(paths["test"])
        test_features_by_encoder[encoder] = features
        print(f"[vlm] {encoder}: test features {features.shape} (RAW_SPACE, from {vlm_dir})")
    else:
        raise ValueError(
            f"Unknown encoder_kind={kind!r} for encoder={encoder!r} -- "
            "evaluate_al_sampler.ipynb only knows how to load 'dinov2' and 'vlm'"
        )

In [ ]:
accuracy = {}

for run_name in RUN_NAMES:
    encoder = run_encoder[run_name]
    test_features = test_features_by_encoder[encoder]
    accuracy[run_name] = {}
    for budget in budgets:
        checkpoint = checkpoint_dir / f"{run_name}_probe_budget_{budget}.pt"
        probe = load_probe(str(checkpoint), device)
        probe_meta = torch.load(str(checkpoint), map_location="cpu", weights_only=False)
        ft_cfg = (probe_meta.get("metadata") or {}).get("final_train_cfg") or {}
        assert not ft_cfg.get("use_lora", False), (
            f"{run_name} budget={budget} was trained with LoRA (final_train_cfg="
            f"{ft_cfg}). Its probe reads features from the adapted encoder, which "
            "this notebook cannot rebuild from the base checkpoint alone -- the "
            "run's own reported metrics (in its _results.pt, computed against "
            "the adapted encoder) are the valid numbers for it. Drop it from "
            "RUN_NAMES here."
        )
        del probe_meta
        assert probe.fc.in_features == test_features.shape[1], (
            f"{run_name} budget={budget}: probe expects {probe.fc.in_features}-d "
            f"features (encoder={encoder!r}) but test_features is "
            f"{test_features.shape[1]}-d -- wrong encoder resolved for this run"
        )
        accuracy[run_name][budget] = evaluate_probe(
            probe, test_features, test_labels, device, verbose=False
        )[0]
        del probe

header = "budget".ljust(9) + "".join(f"{name:>26}" for name in RUN_NAMES)
print(header)
for budget in budgets:
    row = "".join(f"{accuracy[name][budget]:>26.4f}" for name in RUN_NAMES)
    print(f"{budget:<9}{row}")

In [ ]:
for run_name in RUN_NAMES:
    curve = accuracy[run_name]
    if len(curve) < 4:
        print(f"[PALM] {run_name}: need >= 4 budgets, got {len(curve)}")
        continue
    try:
        params = palm_evaluate(budgets=list(curve), accuracies=list(curve.values()))
    except (RuntimeError, ValueError) as error:
        print(f"[PALM] {run_name}: fitting failed: {error}")
        continue
    print(format_palm_report(params, run_name, DATASET))